# 06 — ANBIMA classes

**The question:** *Where did the Brazilian fund industry's money go this year,
by ANBIMA class — and which classes are shrinking?*

Everything else in these notebooks is a **fund**. This one is not: ANBIMA's
*Boletim de Fundos de Investimento* publishes **industry aggregates** — AUM, net
flows, returns and fund counts per class and per ANBIMA type.

That difference has one consequence worth stating before anything else:

> **No fund in this warehouse is mapped to an ANBIMA class.** CVM's `classe` is
> CVM's taxonomy, not ANBIMA's. There is no join from these rows to `fund_nav`
> or `panel`, and none may be invented from a name.

Endpoint: `anbima_classes`.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
COVERAGE = as_of("anbima_classes")

The Boletim is monthly and lands about a month behind, so `as_of` here is
typically a month or two behind the quote tape. Say which month you are quoting.

## Three levels, and they do not stack

`level` picks the grain:

* `"category"` (the default) — the ANBIMA **classes**.
* `"type"` — the ANBIMA **types** underneath a class.
* `"total"` — the **industry** total.

They are three aggregations of the same industry, not three tiers to add
together.

In [ ]:
MONTH = COVERAGE.loc["anbima_classes", "as_of"]
print(f"quoting the {MONTH} Boletim\n")

cats = pd.DataFrame(silo.anbima_classes(metric="pl_brl_mm",
                                        start=MONTH, end=MONTH))
total = pd.DataFrame(silo.anbima_classes(metric="pl_brl_mm", level="total",
                                         start=MONTH, end=MONTH))

print(cats[["category", "level", "metric", "value", "unit"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
print()
print(f"sum of classes : {cats['value'].sum():>14,.1f} (R$ mm)")
print(f"industry total : {total['value'].iloc[0]:>14,.1f} (R$ mm)")
print(f"difference     : {cats['value'].sum() - total['value'].iloc[0]:>14,.1f} (R$ mm)")

The class sum and the published industry total **do not match exactly**, and
neither is adjusted to make them. ANBIMA publishes both figures; a fund can sit
in a class that the total scopes differently, and the Boletim is served as
found.

Reconciling them would mean choosing which number to overwrite. Reporting the
difference costs one line and leaves the reader able to decide.

## `unit` is not decoration — never mix two of them

Every row carries a `unit`: `brl_mm` (R$ **millions**, as published), `pct`
(percentage points) or `count`. Averaging a `pct` row into a `brl_mm` row
produces a number with no meaning, and nothing in the shape of the data stops
you.

In [ ]:
everything = pd.DataFrame(silo.anbima_classes(start=MONTH, end=MONTH))
print(everything.groupby(["unit", "metric"]).size()
      .to_frame("rows").to_string())

## Where the money went

`captacao_liquida_ytd_brl_mm` is net flow year to date, in R$ millions, as
ANBIMA published it. This is a **reported aggregate**, not a sum of the fund
rows in this warehouse — the two are different populations and are never
reconciled here.

In [ ]:
flows = pd.DataFrame(silo.anbima_classes(metric="captacao_liquida_ytd_brl_mm",
                                         start=MONTH, end=MONTH))
aum = cats.set_index("category")["value"]

view = flows[["category", "value"]].rename(columns={"value": "net_flow_ytd_brl_mm"})
view["aum_brl_mm"] = view["category"].map(aum)
view["flow_over_aum_pct"] = (100 * view["net_flow_ytd_brl_mm"]
                             / view["aum_brl_mm"]).round(2)
view = view.sort_values("net_flow_ytd_brl_mm", ascending=False)

print(f"net flow year to date, {MONTH} Boletim (R$ millions, as published)\n")
print(view.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
print()
print("CAVEAT: unit is brl_mm — these are MILLIONS of reais, not reais.")
print("CAVEAT: flow_over_aum_pct is a local division of two published figures,")
print("        not a served metric.")

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2563eb" if v >= 0 else "#dc2626" for v in view["net_flow_ytd_brl_mm"]]
ax.barh(view["category"], view["net_flow_ytd_brl_mm"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("net flow YTD (R$ millions, as published)")
ax.set_title(f"ANBIMA classes — {MONTH} Boletim")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## Inside a class: `level="type"`

In [ ]:
CLASS = view.iloc[0]["category"]
types = pd.DataFrame(silo.anbima_classes(CLASS, metric="pl_brl_mm",
                                         level="type", start=MONTH, end=MONTH))
print(f"ANBIMA types under {CLASS!r}, {MONTH}\n")
print(types[["type_name", "value", "unit"]]
      .sort_values("value", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
print()
print(f"sum of types  : {types['value'].sum():>12,.1f}")
print(f"class total   : {aum[CLASS]:>12,.1f}")

## A series, and the row cap

`anbima_classes` is one of the five functions that **refuse** above a page and
have **no cursor**. Narrow the window, or pin a category.

In [ ]:
series = pd.DataFrame(silo.anbima_classes(CLASS, metric="pl_brl_mm",
                                          start="2024-01-01", end=MONTH))
series["reference_date"] = pd.to_datetime(series["reference_date"])
s = series.set_index("reference_date")["value"].sort_index()

print(f"{CLASS} AUM, R$ millions, {s.index.min():%Y-%m} .. {s.index.max():%Y-%m}\n")
print(s.tail(12).to_string(float_format=lambda v: f"{v:,.1f}"))

In [ ]:
from silo_client.client import SiloError, SiloOverCap

try:
    silo.anbima_classes(start="2000-01-01")     # every class, every metric, 26 years
except SiloOverCap as exc:
    print("refused (over one page, no cursor):")
    print(" ", exc.body)
except SiloError as exc:
    print(f"{type(exc).__name__}: {exc.body}")
else:
    print("this window happened to fit inside one 1,000-row page")

## An unknown argument is an error, not an empty list

This one is a deliberate design choice and worth knowing. Most of this API
answers `200 []` for "nothing matched" — which is right, because an unknown
ticker and an empty window are genuinely both "no rows".

`anbima_classes` does the opposite: an unknown **category**, **metric** or
**level** raises `22023` and **lists what does exist**. A typo'd class name can
never come back as "nothing published that month".

In [ ]:
try:
    silo.anbima_classes("Renda Fixa Longa")      # not an ANBIMA class name
except SiloError as exc:
    print("refused, with the valid values in the message:\n")
    print(exc.body[:900])

## Where this goes next

* Notebook `02` is the fund-level view of the same industry — and **cannot** be
  joined to these rows. The two answer different questions about different
  populations.
* Notebook `04` is the credit corner of it.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.